# Explore New WFS Services - Using Existing Tooling

This notebook tests new data sources mentioned in the Ondrej/Etienne email using our existing **DataCollector** and **DataHandler**.

## New Data Sources to Test:

### WFS Services (Easy - use DataCollector):
1. **RWS-Legger** - NVO construction data, infrastructure
2. **BKN (Beheerkaart Natuur)** - Nature management map
3. **Vegetatiemonitor** - Actual vegetation (needs permission)
4. **Bodemkaart** - Soil composition (need to find WFS URL from BRO-loket)

### Non-WFS Data (Complex - different APIs):
5. **AIS shipping data** - Need to request from RWS
6. **High water events** - Waterweb API
7. **Water flow** - Waterweb API

This notebook focuses on WFS services first.

## Setup - Import Existing Tooling

In [ ]:
import sys
sys.path.append('..')

# Import existing tooling
from src.data.data_collector import DataCollector
from src.data.schema_wfs_service import WfsService
import src.constants as CONST
import src.config as CONFIG

# Standard libraries
import geopandas as gpd
import pandas as pd
from owslib.wfs import WebFeatureService
from IPython.display import display
import folium

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

print("✅ Imports successful")
print(f"Current WFS services in config: {[s.name for s in CONFIG.KNOWN_WFS_SERVICES]}")

✅ Imports successful
Current WFS services in config: ['land_use', 'building_location', 'vegetation']


## Load Test Region

Pick one prediction square (vlakken_scope) to test all WFS services.

In [2]:
gpkg_path = "../data/phase1_2025-08-14_v1.gpkg"
vlakken_scope = gpd.read_file(gpkg_path, layer="vlakken_scope")

# Pick the first region as our test case
test_region_gdf = vlakken_scope.iloc[0:1]  # Keep as GeoDataFrame
test_region = vlakken_scope.iloc[0]  # Single row for easy access
test_geometry = test_region.geometry

print(f"Test region ID: {test_region.get('id', 'unknown')}")
print(f"Geometry type: {test_geometry.geom_type}")
print(f"CRS: {vlakken_scope.crs}")
print(f"Area: {test_geometry.area:.2f} m²")
print(f"Bounds: {test_geometry.bounds}")
print("\n✅ Test region loaded")

Test region ID: unknown
Geometry type: Polygon
CRS: EPSG:28992
Area: 17215.89 m²
Bounds: (148491.67757709508, 416309.3280199179, 148627.60319764755, 416484.88344275043)

✅ Test region loaded


## Helper Function: Explore WFS Capabilities

Before using DataCollector, let's peek at what layers are available in each WFS service.

In [3]:
def explore_wfs_capabilities(name, url, version="2.0.0"):
    """Connect to WFS and list available layers with filtering."""
    print(f"\n{'='*80}")
    print(f"🔍 EXPLORING: {name}")
    print(f"URL: {url}")
    print(f"{'='*80}\n")
    
    try:
        wfs = WebFeatureService(url, version=version)
        layers = list(wfs.contents.items())
        
        print(f"✅ Connected successfully!")
        print(f"📋 Total layers available: {len(layers)}\n")
        
        # Categorize layers by keywords
        categories = {
            "🌊 Shore/Bank (oever)": [],
            "🏗️ Structures (krib, dam, sluis)": [],
            "🌳 Nature/Vegetation (natuur, veg)": [],
            "🚢 Waterway (vaarweg, water)": [],
            "📐 Profile/Cross-section (profiel)": [],
            "🛡️ Protection/Levee (kering, bescherm)": [],
            "🗺️ Other": []
        }
        
        keywords = {
            "🌊 Shore/Bank (oever)": ["oever", "nevengeul"],
            "🏗️ Structures (krib, dam, sluis)": ["krib", "dam", "sluis", "gemaal", "duiker"],
            "🌳 Nature/Vegetation (natuur, veg)": ["natuur", "vegetatie", "bomen", "bodem"],
            "🚢 Waterway (vaarweg, water)": ["vaarweg", "water", "bodem"],
            "📐 Profile/Cross-section (profiel)": ["profiel", "dwarsprofiel"],
            "🛡️ Protection/Levee (kering, bescherm)": ["kering", "bescherm"],
        }
        
        for layer_name, layer in layers:
            categorized = False
            for category, terms in keywords.items():
                if any(term in layer_name.lower() for term in terms):
                    categories[category].append((layer_name, layer.title))
                    categorized = True
                    break
            if not categorized:
                categories["🗺️ Other"].append((layer_name, layer.title))
        
        # Print categorized layers
        for category, layer_list in categories.items():
            if layer_list:
                print(f"\n{category} ({len(layer_list)} layers):")
                for i, (name, title) in enumerate(layer_list[:5], 1):
                    short_name = name.split(':')[-1] if ':' in name else name
                    print(f"  {i}. {short_name}")
                if len(layer_list) > 5:
                    print(f"  ... and {len(layer_list) - 5} more")
        
        print(f"\n{'='*80}\n")
        
        return wfs, layers
        
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        import traceback
        traceback.print_exc()
        return None, []

print("✅ Helper function defined")

✅ Helper function defined


## 1. Explore RWS Legger

In [4]:
wfs_legger, layers_legger = explore_wfs_capabilities(
    name="RWS Legger",
    url="https://geo.rijkswaterstaat.nl/services/ogc/gdr/rws_legger/ows",
    version="2.0.0"
)


🔍 EXPLORING: RWS Legger
URL: https://geo.rijkswaterstaat.nl/services/ogc/gdr/rws_legger/ows

✅ Connected successfully!
📋 Total layers available: 46


🌊 Shore/Bank (oever) (6 layers):
  1. natuurvriendelijke_oever_lijn_legger
  2. natuurvriendelijke_oever_vlak_legger
  3. natuurvriendelijke_vooroever_legger
  4. nevengeul_strang_legger
  5. oeverconstructie_verticaal_legger
  ... and 1 more

🏗️ Structures (krib, dam, sluis) (8 layers):
  1. dam_legger
  2. duiker_lijn_legger
  3. duiker_vlak_legger
  4. gemaal_legger
  5. in_of_uitwateringssluis_legger
  ... and 3 more

🌳 Nature/Vegetation (natuur, veg) (2 layers):
  1. dwarsprofiel_over_genormeerde_bodem_legger
  2. genormeerd_bodem_legger

🚢 Waterway (vaarweg, water) (12 layers):
  1. andere_dan_primaire_waterkering_beschermingszone_legger
  2. andere_dan_primaire_waterkering_waterstaatswerk_legger
  3. begrenzing_rijksvaarweg_legger
  4. niet_primaire_waterkering_beschermingszone__overig_en_regionaal__legger
  5. niet_primaire_water

### Test RWS Legger with DataCollector

Based on the categories above, let's test specific layers using our existing DataCollector.

In [5]:
# Define which layers to test (update this based on exploration above)
# For now, let's test shore/bank related layers
legger_layers_to_test = [
    "rws_legger:oever_legger",  # Shore/bank layer
    "rws_legger:krib_legger",    # Groynes (affects erosion)
]

# Create WfsService config for RWS Legger
service_legger = WfsService(
    name="rws_legger",
    url="https://geo.rijkswaterstaat.nl/services/ogc/gdr/rws_legger/ows",
    version="2.0.0",
    relevant_layers=legger_layers_to_test
)

# Create DataCollector with test region
print(f"🔄 Creating DataCollector for RWS Legger...")
print(f"   Testing layers: {legger_layers_to_test}")
print(f"   Test region area: {test_geometry.area:.2f} m²\n")

collector_legger = DataCollector(
    source_shape=test_geometry,
    source_epsg_crs=CONST.EPSG_RD,
    wfs_services=[service_legger],
    buffer_in_metres=50  # 50m buffer around test region
)

# Query the WFS
print("📡 Querying WFS service...")
collector_legger.get_data_from_all_wfs()

# Check what we got back
print(f"\n✅ Data collection complete!")
print(f"   Services queried: {list(collector_legger.relevant_geospatial_data.keys())}")

🔄 Creating DataCollector for RWS Legger...
   Testing layers: ['rws_legger:oever_legger', 'rws_legger:krib_legger']
   Test region area: 17215.89 m²



Failed to get data from rws_legger (attempt 1/3): 'rws_legger:oever_legger'. Retrying...
Failed to get data from rws_legger (attempt 2/3): 'rws_legger:oever_legger'. Retrying...
Failed to get data from rws_legger after 3 attempts: 'rws_legger:oever_legger'


📡 Querying WFS service...

✅ Data collection complete!
   Services queried: ['rws_legger']


### Examine RWS Legger Results

In [6]:
# Examine data for each layer
for service_name, layers_data in collector_legger.relevant_geospatial_data.items():
    print(f"\n{'='*80}")
    print(f"SERVICE: {service_name}")
    print(f"{'='*80}\n")
    
    for layer_name, gdf in layers_data.items():
        print(f"\n📊 Layer: {layer_name}")
        
        if gdf.empty:
            print(f"   ⚠️  No features found in test region")
        else:
            print(f"   ✅ Features found: {len(gdf)}")
            print(f"   📐 Geometry type: {gdf.geometry.geom_type.unique()}")
            print(f"   🗺️  CRS: {gdf.crs}")
            print(f"   📋 Columns ({len(gdf.columns)}): {list(gdf.columns)[:10]}")
            
            if len(gdf.columns) > 10:
                print(f"       ... and {len(gdf.columns) - 10} more columns")
            
            print(f"\n   📝 Sample data (first 3 rows):")
            display(gdf.head(3))
        
        print(f"\n{'-'*80}")


SERVICE: rws_legger



## 2. Explore BKN (Beheerkaart Natuur)

In [7]:
wfs_bkn, layers_bkn = explore_wfs_capabilities(
    name="BKN (Beheerkaart Natuur)",
    url="https://geo.rijkswaterstaat.nl/services/ogc/gdr/beheerkaart_nat/ows",
    version="2.0.0"
)


🔍 EXPLORING: BKN (Beheerkaart Natuur)
URL: https://geo.rijkswaterstaat.nl/services/ogc/gdr/beheerkaart_nat/ows

✅ Connected successfully!
📋 Total layers available: 32


🌊 Shore/Bank (oever) (2 layers):
  1. oever_lijnen
  2. oever_vlakken

🌳 Nature/Vegetation (natuur, veg) (1 layers):
  1. waterbodem_vlakken

🚢 Waterway (vaarweg, water) (7 layers):
  1. vaarwegmeubilair_niet_roteerbaar
  2. vaarwegmeubilair_roteerbaar
  3. water_punten
  4. water_vlakken
  5. waterafvoer_lijnen
  ... and 2 more

🛡️ Protection/Levee (kering, bescherm) (3 layers):
  1. markering_lijnen
  2. markering_punten
  3. markering_vlakken

🗺️ Other (19 layers):
  1. beheer_vlakken
  2. exploitatie_vlakken
  3. gebouw_en_installatie_vlakken
  4. groen_lijnen
  5. groen_punten
  ... and 14 more




### Test BKN with DataCollector

In [8]:
# Define which BKN layers to test (update based on exploration above)
# Example - you'll need to pick actual layer names after exploration
bkn_layers_to_test = [
    # Add layer names here after exploring capabilities above
    # e.g., "beheerkaart_nat:some_layer"
]

if not bkn_layers_to_test:
    print("⚠️  No layers specified yet. Update bkn_layers_to_test after exploring capabilities above.")
    print("   Then re-run this cell to test with DataCollector.")
else:
    # Create WfsService config for BKN
    service_bkn = WfsService(
        name="bkn",
        url="https://geo.rijkswaterstaat.nl/services/ogc/gdr/beheerkaart_nat/ows",
        version="2.0.0",
        relevant_layers=bkn_layers_to_test
    )
    
    # Create DataCollector
    print(f"🔄 Creating DataCollector for BKN...")
    print(f"   Testing layers: {bkn_layers_to_test}\n")
    
    collector_bkn = DataCollector(
        source_shape=test_geometry,
        source_epsg_crs=CONST.EPSG_RD,
        wfs_services=[service_bkn],
        buffer_in_metres=50
    )
    
    # Query the WFS
    print("📡 Querying WFS service...")
    collector_bkn.get_data_from_all_wfs()
    
    print(f"\n✅ Data collection complete!")
    print(f"   Services queried: {list(collector_bkn.relevant_geospatial_data.keys())}")

⚠️  No layers specified yet. Update bkn_layers_to_test after exploring capabilities above.
   Then re-run this cell to test with DataCollector.


## 3. Template for Additional WFS Services

Use this template to test other WFS services mentioned in the email:
- Vegetatiemonitor (need permission)
- Bodemkaart (need to find WFS URL)
- Any other WFS services

In [ ]:
# TEMPLATE - Copy and modify this for new WFS services
# 
# # Step 1: Explore capabilities
# wfs_new, layers_new = explore_wfs_capabilities(
#     name="Service Name",
#     url="https://service.url/wfs",
#     version="2.0.0"
# )
#
# # Step 2: Define layers to test
# new_layers_to_test = [
#     "service:layer1",
#     "service:layer2"
# ]
#
# # Step 3: Create WfsService
# service_new = WfsService(
#     name="service_name",
#     url="https://service.url/wfs",
#     version="2.0.0",
#     relevant_layers=new_layers_to_test
# )
#
# # Step 4: Test with DataCollector
# collector_new = DataCollector(
#     source_shape=test_geometry,
#     source_epsg_crs=CONST.EPSG_RD,
#     wfs_services=[service_new],
#     buffer_in_metres=50
# )
# collector_new.get_data_from_all_wfs()
#
# # Step 5: Examine results
# for service_name, layers_data in collector_new.relevant_geospatial_data.items():
#     for layer_name, gdf in layers_data.items():
#         print(f"Layer: {layer_name}, Features: {len(gdf)}")
#         if not gdf.empty:
#             display(gdf.head())

print("📋 Template ready - uncomment and modify for new services")

## Summary & Next Steps

After running this notebook:

1. **Review which layers have data** in your test region
2. **Identify relevant layers** for erosion prediction:
   - Shore/bank features (oever)
   - Structures affecting erosion (krib, dam)
   - Nature/vegetation data
   - Protection zones

3. **Add to config.py**: Update `KNOWN_WFS_SERVICES` in `src/config.py` with new services
4. **Add aggregation rules**: Update `AGGREGATION_COLUMNS` in `src/config.py` to define how to aggregate features
5. **Update DATA_INVENTORY.md**: Document findings and which layers are useful
6. **Test with DataHandler**: See if new features improve model predictions

In [9]:
# Summary of data sources from email
print("="*80)
print("DATA SOURCES FROM EMAIL - STATUS")
print("="*80)
print("\n✅ TESTABLE IN THIS NOTEBOOK (WFS Services):")
print("   1. RWS-Legger - NVO construction, infrastructure")
print("   2. BKN - Nature management map")
print("   3. Vegetatiemonitor - Actual vegetation (⚠️ needs permission from RWS)")
print("   4. Bodemkaart - Soil composition (❓ need WFS URL from BRO-loket)")

print("\n⏳ NOT WFS - NEED DIFFERENT APPROACH:")
print("   5. AIS shipping data - Request from RWS, not WFS")
print("   6. High water events - Waterweb API (GitHub repo)")
print("   7. Water flow/rotation - Waterweb API (GitHub repo)")

print("\n📋 ALREADY IN USE:")
print(f"   {[s.name for s in CONFIG.KNOWN_WFS_SERVICES]}")

print("\n" + "="*80)

DATA SOURCES FROM EMAIL - STATUS

✅ TESTABLE IN THIS NOTEBOOK (WFS Services):
   1. RWS-Legger - NVO construction, infrastructure
   2. BKN - Nature management map
   3. Vegetatiemonitor - Actual vegetation (⚠️ needs permission from RWS)
   4. Bodemkaart - Soil composition (❓ need WFS URL from BRO-loket)

⏳ NOT WFS - NEED DIFFERENT APPROACH:
   5. AIS shipping data - Request from RWS, not WFS
   6. High water events - Waterweb API (GitHub repo)
   7. Water flow/rotation - Waterweb API (GitHub repo)

📋 ALREADY IN USE:
   ['land_use', 'building_location', 'vegetation']

